In [4]:
%%capture
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [6]:
# Load data
names = open('names.txt').read().splitlines()

chars = sorted(list(set(''.join(names))))

vocab = ['.'] + chars
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for s,i in stoi.items()}
V = len(vocab)
print(V)

27


In [9]:
vocab

['.',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [11]:
# Build bigram counts
N = torch.zeros((V,V), dtype=torch.float)

for w in names:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

P = (N+1) / (N+1).sum(1, keepdim=True)

In [12]:
# Dataset for MLP: bigram dataset
xs = []
ys = []
for w in names:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
x = torch.tensor(xs, dtype=torch.long)
y = torch.tensor(ys, dtype=torch.long)
print(x.shape, y.shape)

torch.Size([228146]) torch.Size([228146])


In [13]:
# MLP model
class MLP(nn.Module):
    def __init__(self, V, emb=16, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, emb)
        self.fc1 = nn.Linear(emb, hidden)
        self.fc2 = nn.Linear(hidden, V)
    def forward(self, idx):
        x = self.emb(idx)
        x = torch.tanh(self.fc1(x))
        return self.fc2(x)

model = MLP(V).to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-2)

In [14]:
# Training
x_dev = x.to(device)
y_dev = y.to(device)

for step in range(2000):
    logits = model(x_dev)
    loss = F.cross_entropy(logits, y_dev)
    optim.zero_grad()
    loss.backward()
    optim.step()
    if step % 200 == 0:
        print(step, loss.item())

0 3.3115804195404053
200 2.4559569358825684
400 2.4546046257019043
600 2.4543159008026123
800 2.454197645187378
1000 2.454136848449707
1200 2.4542112350463867
1400 2.454071044921875
1600 2.454057216644287
1800 2.4540531635284424


In [16]:
# Sampling
import torch

def sample(model, max_len=20):
    out = []
    idx = torch.tensor([0], dtype=torch.long).to(device)
    for _ in range(max_len):
        logits = model(idx)
        probs = F.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, num_samples=1).squeeze(dim=-1) # Squeeze the tensor to keep it 1-dimensional
        ch = itos[idx.item()]
        if ch == '.': break
        out.append(ch)
    return ''.join(out)

for _ in range(20):
    print(sample(model))

ryaonn
m
onzea
ngnjarin
jor
de
ch
e
amantsen
kiranzy
ava
riera
marir
a
derilalmaivennadal
mpisiawabeorynigrie
idsandannenai
ai
miyaroies
h
